In [ ]:
%load_ext autoreload
%autoreload 2

%load_ext rich

In [ ]:
import json
import mimetypes
import os
import time
from pathlib import Path
from typing import Iterable

import ollama
import requests
from IPython.display import Markdown
from ollama import ChatResponse
from tqdm import tqdm

from aymurai.utils.json_data import load_json, save_json
from aymurai.utils.yaml_data import load_yaml

In [ ]:
os.environ["OLLAMA_KEEP_ALIVE"] = "0"
print("OLLAMA_KEEP_ALIVE set to 0. Models will unload immediately after use.")

## Documents

In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8899")
ENDPOINT = f"{API_BASE_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "/resources/data/restricted/summarization")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

In [ ]:
doc_path = documents[0]
extracted_document = call_extraction_api(requests.Session(), doc_path)["detail"]
document = "\n".join(extracted_document["document"])
print(document)

In [ ]:
extracted_documents = []

for doc_path in tqdm(documents):
    session = requests.Session()
    extracted_document = call_extraction_api(session, doc_path)
    extracted_documents.append(
        {
            "doc_path": os.path.basename(doc_path),
            **extracted_document,
        }
    )
    session.close()

In [ ]:
len(extracted_documents)

In [ ]:
extracted_documents[-1]

In [ ]:
# Filter out empty extracted documents
extracted_documents = [
    doc for doc in extracted_documents if len(doc["detail"]["document"])
]
len(extracted_documents)

## LLMs

In [ ]:
MODELS = [
    # Llama
    "llama3.1:8b",
    "llama3.2:3b",
    # Gemma
    "gemma3:4b",
    "gemma3n:e4b",
    "gemma3:12b",
    # Qwen
    "qwen3:8b",
    "qwen3:14b",
    # Phi
    "phi3:3.8b",
    "phi4:14b",
    # DeepSeek
    "deepseek-r1:8b",
    "deepseek-r1:14b",
    # GPT
    "gpt-oss:20b",
]

### System Prompts

In [ ]:
system_prompts = load_yaml("/resources/llm/summarization.yml")["system-prompts"]
system_prompts

In [ ]:
# Function to get chat response from the model
def get_chat_response(
    system_prompt: str,
    user_prompt: str,
    model: str = "llama3.2:3b",
    options: dict = {"num_ctx": 16_384, "num_predict": 4096},
) -> ChatResponse:
    """
    Get chat response from the model.

    Args:
        system_prompt (str): The system prompt.
        user_prompt (str): The prompt from the user.
        model (str, optional): The model to use. Defaults to "llama3.2:3b".
        options (dict, optional): The options for the chat.
            Defaults to {"num_ctx": 16_384, "num_predict": 4096}.

    Returns:
        ChatResponse: The response from the chat.
    """
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options=options,
    )

    return response

In [ ]:
iter_documents = iter(extracted_documents)

In [ ]:
document = next(iter_documents)
document = "\n".join(document["detail"]["document"])
print(document)

In [ ]:
# Define system prompt and get chat response
system_prompt = system_prompts["baseline"]

# Get chat response
chat_response = get_chat_response(system_prompt, document)
print(chat_response.message.content)

In [ ]:
# Define system prompt and get chat response
system_prompt = system_prompts["task-specific"]

# Get chat response
chat_response = get_chat_response(system_prompt, document)
print(chat_response.message.content)

In [ ]:
# Define prompt parameters
INFORMATION = """
    - Hechos relevantes (qué ocurrió).
    - Actores clave (quiénes están involucrados).
        * Denunciante(s).
        * Denunciado(s).
        * Testigo(s).
        * Juez(es).
        * Abogado(s).
        * Fiscal(es).
        * Perito(s).
        * Otros (especificar).
    - Cronología (cuándo ocurrieron los hechos y eventos procesales importantes).
    - Lugar(es) (dónde ocurrieron los hechos y eventos procesales importantes).
    - Fundamentos (normas legales, precedentes judiciales, doctrinas aplicadas).
    - Decisión (resultado/medida adoptada).
"""

ENTITIES = """
    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección específica (calle, número, intersección de calles y/o avenidas, código postal, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (país, provincia, estado, ciudad, localidad, barrio, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford, Suzuki, etc.).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados (ej., "M.T.G") y los apodos (ej., "el Gato") también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""


# Define system prompt
system_prompt = system_prompts["template"].format(
    information_to_extract=INFORMATION,
    entities_to_identify=ENTITIES,
)

print(system_prompt)

In [ ]:
# Get chat response
chat_response = get_chat_response(system_prompt, document)
Markdown(chat_response.message.content)

In [ ]:
def evaluate_chat_response(
    system_prompt: str,
    user_prompt: str,
    model: str = "llama3.2:3b",
    options: dict = {"num_ctx": 16_384, "num_predict": 4096},
) -> dict:
    """
    Get chat response from the model and evaluate tokens and runtime metrics.

    Args:
        system_prompt (str): The system prompt.
        user_prompt (str): The prompt from the user.
        model (str, optional): The model to use. Defaults to "llama3.2:3b".
        options (dict, optional): The options for the chat.
            Defaults to {"num_ctx": 16_384, "num_predict": 4096}.

    Returns:
        dict: A dictionary containing:
            - response: The content of the response
            - model: The model name
            - input_tokens: Number of tokens in the prompt (prompt_eval_count)
            - input_duration_ms: Time taken to process input tokens in milliseconds
            - output_tokens: Number of tokens in the output (eval_count)
            - output_duration_ms: Time taken to generate output tokens in milliseconds
            - total_tokens: Total number of tokens processed
            - total_duration_ms: Total duration of the request in milliseconds
            - tokens_per_second: Rate of token generation (output tokens / output duration)
    """
    start = time.time()

    # Get the chat response
    response = get_chat_response(
        user_prompt=user_prompt,
        model=model,
        system_prompt=system_prompt,
        options=options,
    )

    end = time.time()

    # Extract metrics
    input_tokens = (
        response.prompt_eval_count if hasattr(response, "prompt_eval_count") else None
    )
    output_tokens = response.eval_count if hasattr(response, "eval_count") else None
    total_tokens = None

    # If total_tokens is still None but we have input and output tokens
    if total_tokens is None and input_tokens is not None and output_tokens is not None:
        total_tokens = input_tokens + output_tokens

    # Extract durations and convert nanoseconds to milliseconds
    input_duration_ms = None
    if hasattr(response, "prompt_eval_duration"):
        input_duration_ms = response.prompt_eval_duration / 1_000_000  # ns to ms

    output_duration_ms = None
    if hasattr(response, "eval_duration"):
        output_duration_ms = response.eval_duration / 1_000_000  # ns to ms

    model_total_duration_ms = None
    if hasattr(response, "total_duration"):
        model_total_duration_ms = response.total_duration / 1_000_000  # ns to ms

    # Calculate the total duration from our timer (this will include network latency)
    measured_duration_ms = (end - start) * 1000

    # Calculate tokens per second for generation
    tokens_per_second = None
    if (
        output_tokens is not None
        and output_duration_ms is not None
        and output_duration_ms > 0
    ):
        tokens_per_second = output_tokens / (output_duration_ms / 1000)

    return {
        "model": model,
        "options": options,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "chat_response": response.message.content,
        "input_tokens": input_tokens,
        "input_duration_ms": input_duration_ms,
        "output_tokens": output_tokens,
        "output_duration_ms": output_duration_ms,
        "total_tokens": total_tokens,
        "model_duration_ms": model_total_duration_ms,
        "measured_duration_ms": measured_duration_ms,
        "tokens_per_second": tokens_per_second,
        "raw_response": response.model_dump(),  # Include the full response object for reference
    }

In [ ]:
# Test our new evaluation function
test_result = evaluate_chat_response(
    system_prompt=system_prompts["task-specific"],
    user_prompt=document,
)

# Display the metrics
print(f"Response from {test_result['model']}:")
print("-" * 50)
print(test_result["raw_response"])
print("-" * 50)
print(f"Input tokens: {test_result['input_tokens']}")
print(f"Input processing time: {test_result['input_duration_ms']:.2f} ms")
print(f"Output tokens: {test_result['output_tokens']}")
print(f"Output generation time: {test_result['output_duration_ms']:.2f} ms")
print(f"Total tokens: {test_result['total_tokens']}")
print(f"Model reported duration: {test_result['model_duration_ms']:.2f} ms")
print(f"Measured duration: {test_result['measured_duration_ms']:.2f} ms")
print(f"Generation speed: {test_result['tokens_per_second']:.2f} tokens/second")

In [ ]:
errors = []
results = []

In [ ]:
DEVICE = "cuda"

results = load_json(f"./summarization-benchmark-results-{DEVICE}.json")
results[-1]

In [ ]:
len(results)

In [ ]:
for extracted_document in tqdm(
    extracted_documents, total=len(extracted_documents), desc="Evaluating documents"
):
    doc_path = os.path.basename(extracted_document["doc_path"])
    document = "\n".join(extracted_document["detail"]["document"])

    if not document.strip():
        print(f"Skipping empty document: {doc_path}")
        continue

    for model in MODELS:
        for system_prompt_type in system_prompts:
            print(
                f"Evaluating {doc_path} with model {model} and prompt {system_prompt_type}"
            )
            if any(
                r
                for r in results
                if r["doc_path"] == doc_path
                and r["model"] == model
                and r["system_prompt_type"] == system_prompt_type
                and r["device"] == DEVICE
            ):
                print(
                    f"Skipping {doc_path} with model {model} and prompt {system_prompt_type} on {DEVICE} (already evaluated)"
                )
                continue

            system_prompt = system_prompts[system_prompt_type]
            if system_prompt_type == "template":
                system_prompt = system_prompts["template"].format(
                    information_to_extract=INFORMATION,
                    entities_to_identify=ENTITIES,
                )

            try:
                result = evaluate_chat_response(
                    system_prompt=system_prompt,
                    user_prompt=document,
                    model=model,
                )
                result["doc_path"] = doc_path
                result["system_prompt_type"] = system_prompt_type
                result["device"] = DEVICE
                results.append(result)

                save_json(
                    results,
                    f"./summarization-benchmark-results-{DEVICE}.json",
                )

            except Exception as e:
                print(f"Error evaluating {doc_path} with model {model}: {e}")
                errors.append(
                    {
                        "doc_path": doc_path,
                        "model": model,
                        "system_prompt": system_prompt,
                        "device": DEVICE,
                        "error": str(e),
                    }
                )


In [ ]:
len(errors)

In [ ]:
errors